***
<center><h1> Accessing Ocean Data with Python </h1></center>
<br>

### Overview of CMEMS Products

| No | Dataset ID | Variable(s) Used | What It Provides | Spatial Resolution | Time Step |
| :---: | :--- | :--- | :--- | :---: | :---: |
| **1** | `METOFFICE-GLO-SST-L4-REP-OBS-SST` | `analysed_sst` | **Sea surface temperature** from the OSTIA system (multi-sensor infrared satellite SST combined with in-situ data). | 0.05° (~5 km) | Daily |
| **2** | `cmems_obs-mob_glo_phy_my_0.125deg_P1D-m` | `so`, geostrophic currents ($u, v$) | **Salinity and geostrophic currents** from data fusion of satellite observations and in-situ profiles. | 0.125° | Daily |
| **3** | `cmems_obs-wave_glo_phy-swh_my_multi-l4-0.5deg_P1D-i` | `VAVH_INST` | **Instantaneous significant wave height** from a multi-sensor, multi-model sea-state product. | 0.5° | Daily |
| **4** | `cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D` | `CHL` | **Surface chlorophyll-a concentration** — a gap-free, multi-sensor ocean color dataset for biological monitoring. | 4 km | Daily |
| **5** | `cmems_obs-oc_glo_bgc-pp_my_l4-multi-4km_P1M` | `PP` | **Primary production** estimated from ocean color measurements using a photosynthesis model. | 4 km | Monthly |
| **6** | `cmems_obs-wind_glo_phy_my_l4_P1M` | `eastward_wind`, `northward_wind` | **10-meter surface wind vectors** ($u, v$) bias-corrected using scatterometer data against atmospheric models. | 0.125° / 0.25° | Monthly |

<font color='blue'>Let's now import our necessary modules !</font>

In [ ]:
## Import required modules

import cartopy.crs as ccrs  # map projections
import cartopy.feature as cfeature  # geographic features (coastlines, land...)
import matplotlib.dates as mdates
import matplotlib.pyplot as plt  # plotting
import numpy as np  # numerical arrays and math tools
import xarray as xr  # work with NetCDF multidimensional datasets
from scipy.ndimage import label  # label connected components
import copernicusmarine # import the Toolbox
import json
import getpass
import os

# To avoid warning messages
import warnings
warnings.filterwarnings("ignore")


<font color='blue'>Lets **log in** to activate your credentials and unlock access to the API.</font>

In [ ]:
copernicusmarine.login()

3.2.1 Remote Data Access (Streaming without Downloading)

In [ ]:
### 3.2.1 Remote Data Access (Streaming without Downloading)

# Set your Copernicus Marine Credentials
#USERNAME = "your_username"
#PASSWORD = "your_password"

# Define Product ID and Dataset ID dictionary for SST OSTIA
product_id = {"sst_ostia": "METOFFICE-GLO-SST-L4-REP-OBS-SST"}

dataset_id = {
    "sst_ostia_daily": "METOFFICE-GLO-SST-L4-REP-OBS-SST"  # Using daily SST dataset
}

product_id_to_retrieve = product_id.get("sst_ostia")
dataset_id_to_retrieve = dataset_id.get("sst_ostia_daily")

print(f"Product id to retrieve: {product_id_to_retrieve}")
print(f"Dataset id to retrieve: {dataset_id_to_retrieve}")

In [ ]:
# Inspect Product Metadata (Updated for copernicusmarine v1.x+)
metadata = copernicusmarine.describe(
    contains=[product_id_to_retrieve]
)
metadata

In [ ]:
def get_metadata(dataset_id):
    # Mengambil metadata dari copernicusmarine
    catalogue = copernicusmarine.describe(contains=[dataset_id])

    # Mengubah objek CopernicusMarineCatalogue menjadi dictionary biasa
    if hasattr(catalogue, "to_dict"):
        data = catalogue.to_dict()
    elif hasattr(catalogue, "dict"):
        data = catalogue.dict()
    else:
        # Fallback untuk tipe data pydantic/custom object
        data = json.loads(catalogue.model_dump_json())

    # Menyimpan dictionary ke file JSON
    output_filename = f"metadata_{dataset_id}.json"
    with open(output_filename, "w", encoding="utf-8") as json_file:
        json.dump(data, json_file, indent=4, default=str)

    print(f"File metadata saved at ./{output_filename}")
    return data


# Fetch and save metadata to local JSON file
get_metadata(product_id_to_retrieve)

In [ ]:
# Stream dataset directly into memory
sst_ostia_stream = copernicusmarine.open_dataset(
    dataset_id=dataset_id_to_retrieve,
    #username=USERNAME,
    #password=PASSWORD
)
sst_ostia_stream

In [ ]:
Direct File Downloading (Subset & Download)

In [ ]:
# Set your credentials
#USERNAME = "your_username"
#PASSWORD = "your_password"

# Define output directory and file details
output_dir = "./data" # directory downloaded data
output_file = "sst_indonesia_202001.nc" # filename
dataset_id_to_retrieve = "METOFFICE-GLO-SST-L4-REP-OBS-SST" #copernicus dataset id that we want to download

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

print("Starting subset and download process...")

# Download subsetted NetCDF file
copernicusmarine.subset(
    dataset_id=dataset_id_to_retrieve,
    variables=["analysed_sst"],
    minimum_longitude=80,
    maximum_longitude=160,
    minimum_latitude=-20,
    maximum_latitude=20,
    start_datetime="2020-01-01T00:00:00",
    end_datetime="2020-01-10T00:00:00",
    output_filename=output_file,
    output_directory=output_dir,
    #username=USERNAME,
    #password=PASSWORD,
    force_download=True,  # Parameter yang diperbarui untuk menimpa file lama
)

print(f"Download completed! File saved at: {output_dir}/{output_file}")

In [ ]:
# 1. Displaying the structure of the REMOTELY STREAMED dataset (l4ds)
print("=== Streamed Dataset Structure ===")
sst_ostia_stream

In [ ]:
# 2. Displaying the structure of the LOCALLY DOWNLOADED dataset
local_file_path = os.path.join("data", "sst_indonesia_202001.nc")

# Open dataset using xarray
sst_ostia_download = xr.open_dataset(local_file_path)

print("=== Downloaded Local Dataset Structure ===")
sst_ostia_download

Pre-processing: Removing Single-Dimensional Axes with

In [ ]:
# Remove single-dimensional axes (length = 1) from the local dataset
ds_local_clean = sst_ostia_download.squeeze()

# Display the cleaned dataset
print("=== Cleaned Local Dataset (Squeezed) ===")
ds_local_clean

Calculate SST average & and plotting maps

In [ ]:
# 1. Define Indonesian bounding box
lon_min, lon_max = 80, 160
lat_min, lat_max = -20, 20

time_start = "2020-01-01"
time_end = "2020-01-31"

# 2. Subset the streamed dataset (sst_ostia_stream) geographically using .sel()
# sst_indo = sst_ostia_stream['analysed_sst'].sel(
#     longitude=slice(lon_min, lon_max),
#     latitude=slice(lat_min, lat_max)
# )

sst_indo = sst_ostia_stream['analysed_sst'].sel(
    time=slice(time_start, time_end),
    longitude=slice(lon_min, lon_max),
    latitude=slice(lat_min, lat_max)
)

# 3. Compute mean SST along the time dimension and convert Kelvin to Celsius
# calculate mean without add compute (try this first)
#sst_mean_indo = sst_indo['analysed_sst'].mean(dim='time') - 273.15

# calculate mean with add compute
from dask.diagnostics import ProgressBar
with ProgressBar():
    sst_mean_indo = (sst_indo.mean(dim="time") - 273.15).compute()

# Display the resulting 2D mean SST object
print("=== Mean SST Field (Indonesian Region) ===")
sst_mean_indo

In [ ]:
# print the min and max value of sst data
print("Min temp:", float(sst_mean_indo.min()))
print("Max temp:", float(sst_mean_indo.max()))
print("Shape data:", sst_mean_indo.shape)

Plotting a Temperature Map

In [ ]:
# Step 1: Create figure and Map Axis
fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

In [ ]:
# Step 2.:Add basic geographic featuresax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.gridlines(draw_labels=True)
ax.set_extent([lon_min, lon_max, lat_min, lat_max])

fig

In [ ]:
# Reusable function to add geographic features to a map axis
def basic_geo_features(ax, lon_min=None, lon_max=None, lat_min=None, lat_max=None):
    # Add land mask and coastlines
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=2)
    ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.5, zorder=2)
    
    # Add gridlines with lat/lon labels
    gl = ax.gridlines(draw_labels=True, crs=ccrs.PlateCarree(), linestyle="--", alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    
    # Set spatial extent
    if None not in (lon_min, lon_max, lat_min, lat_max):
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

In [ ]:
# Step 3: Plot the SST mean field using pcolormesh
# Note: sst_mean_indo is a 2D array (latitude x longitude)
im = ax.pcolormesh(
    sst_mean_indo.longitude.data,
    sst_mean_indo.latitude.data,
    sst_mean_indo.data,
    cmap="turbo",       # Perceptually uniform colormap suited for ocean temperatures
    vmin=24,           # Min temperature limit in °C
    vmax=31,           # Max temperature limit in °C
    transform=ccrs.PlateCarree()
)

fig

In [ ]:
# Step 4: Add Colorbar and Title (with Dynamic Time Period)
cbar = plt.colorbar(im, ax=ax, label="Mean Sea Surface Temperature (°C)", shrink=0.7, pad=0.03)

# Add Title with explicit time range
ax.set_title(
    f"Mean Sea Surface Temperature (SST) - Indonesian Seas\n"
    f"Streamed CMEMS OSTIA Product ({time_start} to {time_end})", 
    fontsize=12, 
    fontweight='bold'
)

fig

We will combine all these steps in a single code cell, as shown below:

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

# Step 1: Create figure and Map Axis
fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.PlateCarree())

# Step 2: Apply basic geographic features (off South Java)
# Fungsi basic_geo_features yang sudah didefinisikan sebelumnya
basic_geo_features(ax, lon_min, lon_max, lat_min, lat_max)

# Step 3: Compute SST Mean Field off South Java & Plot using imshow
# Menggunakan variabel analysed_sst dari dataset ds lokal kita
sst_mean_sj = ds["analysed_sst"].mean(dim="time").compute()

im = ax.imshow(
    sst_mean_sj.data,
    extent=[
        sst_mean_sj.longitude.min(),
        sst_mean_sj.longitude.max(),
        sst_mean_sj.latitude.min(),
        sst_mean_sj.latitude.max(),
    ],
    origin="lower",
    cmap="turbo",
    vmin=25,
    vmax=31,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

# Step 4: Add Colorbar and Title
cbar = plt.colorbar(
    im,
    ax=ax,
    label="Mean Sea Surface Temperature (°C)",
    shrink=0.75,
    pad=0.03,
)

# Ambil rentang tahun secara otomatis dari dataset lokal
start_yr = str(ds.time.dt.year.min().data)
end_yr = str(ds.time.dt.year.max().data)

ax.set_title(
    f"Mean Sea Surface Temperature (SST) off South Java\n"
    f"CMEMS OSTIA Dataset ({start_yr} - {end_yr})",
    fontsize=12,
    fontweight="bold",
    pad=12,
)

plt.show()

SST Timeseries Analysis

In [ ]:
# 1. Download the data (2000-2005)
print("Downloading Java Sea SST subset from CMEMS...")
copernicusmarine.subset(
    dataset_id="METOFFICE-GLO-SST-L4-REP-OBS-SST",
    variables=["analysed_sst"],
    minimum_longitude=106.0,
    maximum_longitude=116.0,
    minimum_latitude=-6.5,
    maximum_latitude=-3.0,
    start_datetime="2000-01-01T00:00:00",
    end_datetime="2010-12-31T00:00:00",
    output_filename="java_sea_sst_2000_2010.nc",
    output_directory="./data",
    force_download=True,
)

In [ ]:
# read the data via streaming

# Define Java Sea region & central point
# lon_point, lat_point = 110.0, -5.0
# lon_min_js, lon_max_js = 106.0, 116.0
# lat_min_js, lat_max_js = -6.5, -3.0

# Define 5-year target period
# start_year = "2000-01-01"
# end_year = "2005-12-31"

# -----------------------------------------------------------------
# Versi 1: Single Point Extraction (2000-2005)
# -----------------------------------------------------------------
# print("Fetching Point Timeseries data (2000-2005)...")
# with ProgressBar():
#     ts_point = (
#         sst_ostia_stream["analysed_sst"]
#         .sel(time=slice(start_year, end_year))
#         .sel(longitude=lon_point, latitude=lat_point, method="nearest")
#         - 273.15
#     ).compute()

# print("Fetching Point Timeseries data (Optimized)...")
# with ProgressBar():
#     ts_point = (
#         sst_ostia_stream["analysed_sst"]
#         .sel(longitude=lon_point, latitude=lat_point, method="nearest")
#         .sel(time=slice(start_year, end_year, 10))
#         - 273.15
#     ).compute()

# # -----------------------------------------------------------------
# # Versi 2: Spatial Mean Over Java Sea Region (2000-2005)
# # -----------------------------------------------------------------
# print("Fetching Spatial Mean Timeseries data (2000-2005)...")
# with ProgressBar():
#     ts_spatial = (
#         sst_ostia_stream["analysed_sst"]
#         .sel(time=slice(start_year, end_year))
#         .sel(
#             longitude=slice(lon_min_js, lon_max_js),
#             latitude=slice(lat_min_js, lat_max_js),
#         )
#         .mean(dim=["longitude", "latitude"])
#         - 273.15
#     ).compute()

# print("5-Year Timeseries extraction completed successfully!")

In [ ]:
# 2. Open the NetCDF file
ds_js = xr.open_dataset("./data/java_sea_sst_2000_2010.nc")

# 3. Calculate the timeseries
# Versi 1: Point Extraction (110.0°E, 5.0°S)
ts_point = (
    ds_js["analysed_sst"].sel(longitude=110.0, latitude=-5.0, method="nearest")
    - 273.15
)

# Versi 2: Spatial Mean (Java Sea)
ts_spatial = ds_js["analysed_sst"].mean(dim=["longitude", "latitude"]) - 273.15

print("Timeseries extraction completed instantly!")

Plotting Timeseries of SST

In [ ]:
import matplotlib.pyplot as plt

# Extract time coordinates and values
dates_point = ts_point.time.data
vals_point = ts_point.data

dates_spatial = ts_spatial.time.data
vals_spatial = ts_spatial.data

# Create plot
fig, ax = plt.subplots(figsize=(12, 5))

# Plot both versions
ax.plot(
    dates_point,
    vals_point,
    color="tab:blue",
    linewidth=1.2,
    alpha=0.8,
    label="Point (110.0°E, 5.0°S)",
)
ax.plot(
    dates_spatial,
    vals_spatial,
    color="darkorange",
    linewidth=1.5,
    label="Spatial Mean (Java Sea)",
)

# Customize graph
ax.grid(True, linestyle="--", alpha=0.6)
ax.set_xlabel("Time", fontsize=11)
ax.set_ylabel("SST (°C)", fontsize=11)
ax.set_title(
    "Daily Sea Surface Temperature (SST) in the Java Sea\nSingle Point vs Spatial Average",
    fontsize=12,
    fontweight="bold",
)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

Resampling Data to Monthly Means

In [ ]:
# Resample daily data to monthly means using .resample(time="1ME")
ts_spatial_m = ts_spatial.resample(time="1ME").mean()

dates_m = ts_spatial_m.time.data
vals_m = ts_spatial_m.data

# Plot Monthly Mean Timeseries
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    dates_m,
    vals_m,
    color="crimson",
    marker="o",
    markersize=4,
    linewidth=1.5,
    label="Monthly Mean SST",
)

ax.grid(True, linestyle="--", alpha=0.6)
ax.set_xlabel("Time", fontsize=11)
ax.set_ylabel("SST (°C)", fontsize=11)
ax.set_title(
    "Monthly Spatially Averaged SST in the Java Sea",
    fontsize=12,
    fontweight="bold",
)
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

Resampling Data to Annual Means

In [ ]:
# Resample daily data to yearly means using .resample(time="1YE")
ts_spatial_y = ts_spatial.resample(time="1YE").mean()

dates_y = ts_spatial_y.time.data
vals_y = ts_spatial_y.data

# Plot Yearly Mean Timeseries
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    dates_y,
    vals_y,
    color="navy",
    marker="s",
    linewidth=2,
    markersize=6,
    label="Annual Mean SST",
)

ax.grid(True, linestyle="--", alpha=0.6)
ax.set_xlabel("Time", fontsize=11)
ax.set_ylabel("SST (°C)", fontsize=11)
ax.set_title(
    "Yearly Mean SST in the Java Sea", fontsize=12, fontweight="bold"
)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

Hands-On Exercise: Explore Other Ocean Variables!

Now it's your turn to apply what you have learned! Oceanography is multi-disciplinary, and SST is just one piece of the puzzle. Using the exact same workflow (`copernicusmarine` streaming/download, `xarray` selection, `.mean()`, and `cartopy` plotting), try exploring one or more of the following CMEMS products across Indonesian waters: